# Project 2: Complex Contagion
    Abraham Harris

### Setup and Imports

In [1]:
import numpy as np
import networkx as nx
import plotting_utilities as pu
import matplotlib.pyplot as plt
from scipy.io import mmread

### Experiment Harness and Helper Functions

In [ ]:
class Agent:
    def __init__(self, identifier, state="B"):
        """
        Each agent is identified by an integer and corresponds to that node in graph G.
        Agents are in baseline state B or adopted state A.
        """
        assert state in set(["A", "B"])
        self.identifier = identifier
        self.state = state
        

class Population:
    def __init__(self, G, A_count, B_count):
        """Accept a graph G. Set node states as "A" or "B"."""
        self.G = G
        n = self.G.number_of_nodes()
        self.A = A_count
        self.B = B_count

        # Make agents and match them to graph nodes
        self.agents = []
        counter = 0
        indices = np.arange(n) 
        self.index_to_agent = dict()
        np.random.shuffle(indices) # Prevent all nodes of low index having same state
        for _ in range(A_count):
            self.agents.append(Agent(int(indices[counter]), state="A"))
            self.G.nodes[indices[counter]]["state"] = "A"
            self.index_to_agent[indices[counter]] = self.agents[-1]
            counter += 1
        for _ in range(B_count):
            self.agents.append(Agent(int(indices[counter]), state="B"))
            self.G.nodes[indices[counter]]["state"] = "B"
            self.index_to_agent[indices[counter]] = self.agents[-1]
            counter += 1
        np.random.shuffle(self.agents)

    def step_agents(self):
        """Update states of all agents."""
        for agent in self.agents:
            if agent.state == "B":
                neighbors = list(self.G.neighbors(agent.identifier))
                for i in neighbors:
                    if self.G.nodes[i]["state"] == "I":
                        d = self.agents[i].days_spent_infectious
                        # Infectiousness of neighbor i
                        p_i_inf = (p_1c/(1 - p_1c))*np.exp(beta*(d**3 - 1)) / (1 + (p_1c/(1 - p_1c))*np.exp(beta*(d**3 - 1)))
                        r = np.random.uniform()
                        # Determine whether infected by neighbor i
                        if r < p_i_inf:
                            agent.state = "E"
                            self.G.nodes[agent.identifier]["state"] = "E"
                            self.changed_agents.append(agent.identifier)
                            self.S -= 1
                            self.E += 1
                            break